In [1]:
from collections import defaultdict
import json

import wandb


In [2]:
from wandb.old.summary import SummarySubDict

api = wandb.Api()

def dictify(d):
    if isinstance(d, SummarySubDict):
        return {k: dictify(v) for k, v in d.items()}
    if isinstance(d, dict):
        return {k: dictify(v) for k, v in d.items()}
    elif isinstance(d, list):
        return [dictify(v) for v in d]
    else:
        return d

In [4]:
runs = api.runs(
    "samoed-roman/new_autointent_few_shot2", 
    filters={"displayName": "final_metrics"},
)


In [10]:
results = defaultdict(lambda: defaultdict(dict))
for i, run in enumerate(runs):
    print(f"Run {i}: {run.name} - {run.group} - {run.config}")
    ds, few_shot = run.group.split("_")
    few_shot = int(few_shot) if few_shot != "None" else "full"
    results[ds][few_shot] = {
        "config": dictify(run.config),
        **{
            k: dictify(v)
            for k, v in run.summary.items() if k not in ["_wandb", "_timestamp", "_step"]
        },
    }

Run 0: final_metrics - DeepPavlov/hwu64_4 - {'regex': [], 'scoring': [{'metrics': {'scoring_f1': 0.6421417490424406, 'scoring_recall': 0.6616657954042386, 'scoring_roc_auc': 0.9576753340343094, 'scoring_accuracy': 0.662165363657901, 'scoring_precision': 0.654022267381399}, 'metric_name': 'scoring_accuracy', 'module_name': 'knn', 'metric_value': 0.662165363657901, 'module_params': {'k': 8, 'weights': 'closest', 'embedder_config': {'device': None, 'freeze': True, 'use_cache': True, 'batch_size': 32, 'model_name': 'sentence-transformers/all-MiniLM-L6-v2', 'sts_prompt': None, 'query_prompt': None, 'cluster_prompt': None, 'default_prompt': None, 'passage_prompt': None, 'tokenizer_config': {'padding': True, 'max_length': None, 'truncation': True}, 'classifier_prompt': None, 'trust_remote_code': False, 'similarity_fn_name': 'cosine'}}, 'module_dump_dir': None}, {'metrics': {'scoring_f1': 0.6323564251869715, 'scoring_recall': 0.6656003409713505, 'scoring_roc_auc': 0.9729306124369612, 'scoring_

In [12]:
results.keys()

dict_keys(['DeepPavlov/hwu64', 'DeepPavlov/minds14', 'DeepPavlov/snips'])

In [16]:
import pandas as pd

result_df = pd.concat({
    dataset: pd.DataFrame(models).T 
    for dataset, models in results.items()
})

In [21]:
result_df.rename_axis(["dataset", "few_shot"]).to_csv("few_shot_results.csv")

In [ ]:
runs = api.runs(
    "samoed-roman/AutoML-Eval",
)

In [29]:
results = defaultdict(lambda: defaultdict(lambda: defaultdict(dict)))
for run in runs:
    print(f"Run {run.name} - {run.config}")
    metrics = {}
    for k, v in run.summary.items():
        if k in ["accuracy", "_runtime"]:
            metrics[k] = v
        if k in ["macro avg", "weighted avg"]:
            for sub_k, sub_v in v.items():
                metrics[f"{k}_{sub_k}"] = sub_v
    results[run.config["dataset"]][run.config["framework"]][run.config.get("few_shot", "full")] = {
        "config": dictify(run.config),
        **metrics,
    }

Run eval-DeepPavlov/clinc150-fedot - {'dataset': 'DeepPavlov/clinc150', 'framework': 'fedot'}
Run eval-DeepPavlov/clinc150-lama - {'dataset': 'DeepPavlov/clinc150', 'framework': 'lama'}
Run eval-DeepPavlov/massive-lama - {'dataset': 'DeepPavlov/massive', 'framework': 'lama'}
Run eval-DeepPavlov/minds14-lama - {'dataset': 'DeepPavlov/minds14', 'framework': 'lama'}
Run eval-DeepPavlov/snips-lama - {'dataset': 'DeepPavlov/snips', 'framework': 'lama'}
Run eval-DeepPavlov/hwu64-lama - {'dataset': 'DeepPavlov/hwu64', 'framework': 'lama'}
Run eval-DeepPavlov/clinc150-glueon - {'dataset': 'DeepPavlov/clinc150', 'framework': 'glueon'}
Run eval-DeepPavlov/massive-fedot - {'dataset': 'DeepPavlov/massive', 'framework': 'fedot'}
Run eval-DeepPavlov/minds14-fedot - {'dataset': 'DeepPavlov/minds14', 'framework': 'fedot'}
Run eval-DeepPavlov/snips-fedot - {'dataset': 'DeepPavlov/snips', 'framework': 'fedot'}
Run eval-DeepPavlov/massive-glueon - {'dataset': 'DeepPavlov/massive', 'framework': 'glueon'}


In [30]:
results_df = pd.concat({
    dataset: pd.concat({
        framework: pd.DataFrame(models).T 
        for framework, models in frameworks.items()
    })
    for dataset, frameworks in results.items()
})

In [31]:
results_df.index.names = ["dataset", "framework", "few_shot"]
results_df.to_csv("automl_eval_results.csv")